In [0]:
%sql
USE CATALOG shopsphere;

CREATE TABLE IF NOT EXISTS shopsphere.silver.clickstream(
    event_id LONG,
   customer_id DOUBLE,
   session_id STRING,
   event_time TIMESTAMP,
   event_type STRING,
   product_id INT,
   page STRING,
   device_type STRING
);

CREATE TABLE IF NOT EXISTS shopsphere.quarantine.clickstream(
    event_id LONG,
   customer_id DOUBLE,
   session_id STRING,
   event_time TIMESTAMP,
   event_type STRING,
   product_id INT,
   page STRING,
   device_type STRING
)
USING DELTA;

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

df = spark.read.table("shopsphere.bronze.clickstream")

In [0]:
#remove duplicates
df = df.dropDuplicates(["event_id"])


In [0]:
#validate event_time
df_valid = df.filter(col("event_time").isNotNull())

df_quarantine = df.filter(
    col("event_time").isNull()).withColumn("validation_status", lit("invalid_event_time"))


#quarantine rejected rows from order_date
quarantine_table = DeltaTable.forName(spark,"shopsphere.quarantine.clickstream")

quarantine_table.alias("target").merge(df_quarantine.alias("source"),
                                       "target.event_id = source.event_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#Standardize event types
df_valid = df_valid.withColumn("event_type", initcap(trim(col("event_type"))))


In [0]:
#derive event date & hour
df_valid = df_valid.withColumn("event_date", date_format(col("event_time"), "yyyy-MM-dd")).withColumn("event_hour", hour(col("event_time")))


In [0]:
#write transformed data to silver table
silver_table = DeltaTable.forName(spark,"shopsphere.silver.clickstream")

silver_table.alias("target").merge(df_valid.alias("source"),
                                       "target.event_id = source.event_id").\
                                        whenNotMatchedInsertAll().\
                                        withSchemaEvolution().\
                                        execute()

In [0]:
#Anonymous session
%sql

SELECT * FROM shopsphere.silver.clickstream
WHERE customer_id IS NULL